# LSTM (Long Short-Term Memory) — Sentiment Classification on IMDB

This notebook demonstrates how to build an **LSTM** for **sentiment analysis** using **PyTorch**, with a focus on building a **custom Dataset and DataLoader** from scratch.

**What you will learn:**
- How to build a custom `torch.utils.data.Dataset` class
- How to write a custom `collate_fn` for variable-length sequences
- How to create a `DataLoader` with your custom components
- How to build a vocabulary from raw text
- How LSTMs process sequential data and maintain long-term memory

**Dataset:** IMDB Movie Reviews — 50,000 reviews labeled as positive or negative sentiment.

The dataset is loaded via HuggingFace `datasets` library (auto-downloads).

## Step 1 — Install Required Packages

We need `torch` for modeling, `datasets` from HuggingFace to load the IMDB data.

In [9]:
!pip install torch datasets --quiet

## Step 2 — Imports & Device Setup

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Step 3 — Load the IMDB Dataset

We use HuggingFace `datasets` to download the IMDB dataset. It comes pre-split into train and test sets, each with 25,000 reviews.

Each sample has:
- `text`: the raw review string
- `label`: 0 (negative) or 1 (positive)

In [11]:
# Load the IMDB dataset
raw_dataset = load_dataset("imdb")

print(f"Train samples: {len(raw_dataset['train'])}")
print(f"Test samples: {len(raw_dataset['test'])}")

# Look at a sample
sample = raw_dataset['train'][0]
print(f"\nSample review (first 200 chars): {sample['text'][:200]}...")
print(f"Label: {sample['label']} ({'positive' if sample['label'] == 1 else 'negative'})")

Train samples: 25000
Test samples: 25000

Sample review (first 200 chars): I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...
Label: 0 (negative)


## Step 4 — Build a Vocabulary from Scratch

Neural networks need **numbers**, not words. So we need to:
1. **Tokenize**: Split text into words
2. **Build a vocabulary**: Map each unique word to a unique integer index

We'll keep only the top `vocab_size` most frequent words to manage memory. Unknown words get mapped to a special `<UNK>` token, and we use `<PAD>` for padding shorter sequences.

**Why build it from scratch?** To understand the full pipeline. In production, you'd use a tokenizer from HuggingFace or torchtext.

In [12]:
def simple_tokenizer(text):
    """Lowercase and split text into words, removing non-alphanumeric characters."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.split()

# Count word frequencies across the entire training set
word_counts = Counter()
for sample in raw_dataset['train']:
    tokens = simple_tokenizer(sample['text'])
    word_counts.update(tokens)

print(f"Total unique words: {len(word_counts)}")
print(f"Top 10 words: {word_counts.most_common(10)}")

# Build vocabulary: keep top 20,000 words
VOCAB_SIZE = 20000
PAD_IDX = 0  # Padding token index
UNK_IDX = 1  # Unknown token index

# Start with special tokens
word_to_idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}

# Add top words
for word, count in word_counts.most_common(VOCAB_SIZE - 2):  # -2 for PAD and UNK
    word_to_idx[word] = len(word_to_idx)

idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print(f"\nVocabulary size: {len(word_to_idx)}")
print(f"Index of 'movie': {word_to_idx.get('movie', UNK_IDX)}")
print(f"Index of 'great': {word_to_idx.get('great', UNK_IDX)}")

Total unique words: 120775
Top 10 words: [('the', 334713), ('and', 162228), ('a', 161946), ('of', 145326), ('to', 135042), ('is', 106855), ('in', 93028), ('it', 77101), ('i', 75719), ('this', 75190)]

Vocabulary size: 20000
Index of 'movie': 18
Index of 'great': 85


## Step 5 — Custom Dataset Class

This is the core of this notebook. We'll create a custom `Dataset` that PyTorch's `DataLoader` can work with.

A custom Dataset must implement **3 methods**:
- `__init__`: Store the data and any configuration
- `__len__`: Return the total number of samples
- `__getitem__`: Return a single sample by index (tokenized + numericalized)

Our `IMDBDataset` will:
- Tokenize the raw text into words
- Convert words to integer indices using our vocabulary
- Truncate sequences longer than `max_len` (to save memory and speed up training)

In [13]:
class IMDBDataset(Dataset):
    """
    Custom Dataset for IMDB reviews.
    
    Handles tokenization, numericalization (word → index), and truncation.
    Padding is handled later in the collate_fn (because it depends on the batch).
    """
    
    def __init__(self, hf_dataset, word_to_idx, max_len=256):
        """
        Args:
            hf_dataset: HuggingFace dataset split (e.g., raw_dataset['train'])
            word_to_idx: Dictionary mapping words to integer indices
            max_len: Maximum sequence length (longer reviews are truncated)
        """
        self.texts = hf_dataset['text']
        self.labels = hf_dataset['label']
        self.word_to_idx = word_to_idx
        self.max_len = max_len
        self.unk_idx = word_to_idx.get('<UNK>', 1)
    
    def __len__(self):
        """Return the total number of samples."""
        return len(self.texts)
    
    def __getitem__(self, idx):
        """
        Return a single sample: (token_indices, label)
        
        This is called by the DataLoader for each sample in a batch.
        """
        # Tokenize the review text
        tokens = simple_tokenizer(self.texts[idx])
        
        # Truncate to max_len
        tokens = tokens[:self.max_len]
        
        # Convert words to indices (numericalize)
        # Unknown words get the UNK index
        indices = [self.word_to_idx.get(token, self.unk_idx) for token in tokens]
        
        # Return as tensors
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

# Create dataset instances
MAX_LEN = 256
train_dataset = IMDBDataset(raw_dataset['train'], word_to_idx, max_len=MAX_LEN)
test_dataset = IMDBDataset(raw_dataset['test'], word_to_idx, max_len=MAX_LEN)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Look at a single sample
sample_indices, sample_label = train_dataset[0]
print(f"\nSample token indices shape: {sample_indices.shape}")
print(f"First 10 indices: {sample_indices[:10]}")
print(f"Label: {sample_label.item()} ({'positive' if sample_label.item() == 1 else 'negative'})")

# Decode back to words to verify
decoded = [idx_to_word.get(idx.item(), '<UNK>') for idx in sample_indices[:10]]
print(f"Decoded tokens: {decoded}")

Train dataset size: 25000
Test dataset size: 25000

Sample token indices shape: torch.Size([256])
First 10 indices: tensor([  10, 1533,   10,  237,    1,   36,   56,  393, 1148,   80])
Label: 0.0 (negative)
Decoded tokens: ['i', 'rented', 'i', 'am', '<UNK>', 'from', 'my', 'video', 'store', 'because']


## Step 6 — Custom Collate Function

**Problem:** Reviews have different lengths, but PyTorch tensors need uniform dimensions within a batch.

**Solution:** A custom `collate_fn` that:
1. Receives a list of samples from the Dataset
2. Finds the maximum sequence length in this batch
3. **Pads** shorter sequences with the `<PAD>` token (index 0) so all sequences have the same length
4. Returns properly shaped tensors

**Why pad per-batch instead of globally?** Per-batch padding is more efficient — we only pad to the longest sequence in each batch, not the longest in the entire dataset.

In [14]:
def collate_fn(batch):
    """
    Custom collate function for variable-length sequences.
    
    Args:
        batch: List of (token_indices, label) tuples from the Dataset
    
    Returns:
        padded_sequences: Tensor of shape (batch_size, max_seq_len)
        labels: Tensor of shape (batch_size,)
        lengths: Tensor of shape (batch_size,) — original lengths before padding
    """
    # Separate sequences and labels
    sequences, labels = zip(*batch)
    
    # Get the original length of each sequence
    lengths = torch.tensor([len(seq) for seq in sequences])
    
    # Find the max length in this batch
    max_len = max(len(seq) for seq in sequences)
    
    # Pad each sequence to max_len with PAD_IDX (0)
    padded_sequences = torch.zeros(len(sequences), max_len, dtype=torch.long)  # filled with 0 (PAD)
    for i, seq in enumerate(sequences):
        padded_sequences[i, :len(seq)] = seq
    
    # Stack labels into a tensor
    labels = torch.stack(labels)
    
    return padded_sequences, labels, lengths

print("Custom collate_fn defined!")
print("It pads variable-length sequences to the max length in each batch.")

Custom collate_fn defined!
It pads variable-length sequences to the max length in each batch.


## Step 7 — Create DataLoaders

Now we combine our **custom Dataset** + **custom collate_fn** into `DataLoader` instances.

**DataLoader parameters explained:**
- `batch_size=64`: Process 64 reviews at a time
- `shuffle=True` (train only): Randomize order each epoch — helps the model generalize
- `collate_fn=collate_fn`: Use our custom padding function
- `num_workers=2`: Use 2 background processes for data loading (faster on multi-core CPUs)
- `drop_last=True` (train only): Drop the last incomplete batch for consistent batch sizes

In [15]:
BATCH_SIZE = 64

train_loader = DataLoader(
    dataset=train_dataset,       # Our custom IMDBDataset
    batch_size=BATCH_SIZE,       # 64 reviews per batch
    shuffle=True,                # Randomize training order
    collate_fn=collate_fn,       # Our custom padding function
    num_workers=0,               # 0 for macOS compatibility (use 2+ on Linux)
    drop_last=True               # Drop incomplete last batch
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,               # No need to shuffle test data
    collate_fn=collate_fn,
    num_workers=0
)

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches per epoch: 390
Test batches: 391


## Step 8 — Inspect a Batch

Let's pull one batch from the DataLoader and inspect its structure. This is a crucial debugging step when building custom data pipelines.

In [16]:
# Get one batch
sample_batch_seqs, sample_batch_labels, sample_batch_lengths = next(iter(train_loader))

print("=== Batch Inspection ===")
print(f"Sequences shape: {sample_batch_seqs.shape}")   # (batch_size, max_seq_len_in_batch)
print(f"Labels shape: {sample_batch_labels.shape}")      # (batch_size,)
print(f"Lengths shape: {sample_batch_lengths.shape}")     # (batch_size,)
print(f"\nSequence length range: {sample_batch_lengths.min().item()} to {sample_batch_lengths.max().item()}")
print(f"Padded to: {sample_batch_seqs.shape[1]}")

# Decode first review in the batch (first 20 words)
first_seq = sample_batch_seqs[0]
first_len = sample_batch_lengths[0].item()
decoded_words = [idx_to_word.get(idx.item(), '<UNK>') for idx in first_seq[:20]]
print(f"\nFirst review (first 20 tokens): {' '.join(decoded_words)}")
print(f"Label: {'positive' if sample_batch_labels[0].item() == 1 else 'negative'}")
print(f"Original length: {first_len}")

# Show that padding is at the end
if first_len < sample_batch_seqs.shape[1]:
    print(f"\nTokens at positions [{first_len-2}:{first_len+2}]: {first_seq[first_len-2:first_len+2].tolist()}")
    print("(Notice the 0s after the original length — those are PAD tokens)")

=== Batch Inspection ===
Sequences shape: torch.Size([64, 256])
Labels shape: torch.Size([64])
Lengths shape: torch.Size([64])

Sequence length range: 43 to 256
Padded to: 256

First review (first 20 tokens): the ring was made from the only screenplay hitchcock wrote himself and it deals as many of his earliest pictures
Label: positive
Original length: 256


## Step 9 — Define the LSTM Model

Our LSTM architecture for sentiment classification:

```
Input (token indices)
  → Embedding(vocab_size, embed_dim)      — convert indices to dense vectors
  → LSTM(embed_dim, hidden_dim, layers=2) — process sequence, maintain memory
  → Take last hidden state               — summary of the entire sequence
  → Dropout(0.5)                          — regularization
  → Linear(hidden_dim, 1)                 — binary classification
  → Sigmoid                               — output probability [0, 1]
```

**Key LSTM concepts:**
- **Hidden state (h)**: Short-term memory — summarizes recent inputs
- **Cell state (c)**: Long-term memory — allows information to flow unchanged across many timesteps  
- **Gates**: The LSTM has 3 gates:
  - **Forget gate**: Decides what to remove from cell state
  - **Input gate**: Decides what new information to store
  - **Output gate**: Decides what to output from cell state
- This gating mechanism solves the **vanishing gradient problem** that plagues vanilla RNNs

In [17]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, pad_idx):
        super(LSTMClassifier, self).__init__()
        
        # Embedding layer: converts token indices to dense vectors
        # padding_idx tells the layer to output zeros for PAD tokens
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )
        
        # LSTM layer
        # batch_first=True means input shape is (batch, seq_len, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0,  # Dropout between LSTM layers
            bidirectional=False
        )
        
        # Classification head
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, 1)  # Binary classification
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x, lengths):
        """
        Args:
            x: Padded token indices, shape (batch_size, seq_len)
            lengths: Original lengths before padding, shape (batch_size,)
        """
        # Step 1: Embed tokens
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        
        # Step 2: Pack padded sequences for efficient LSTM processing
        # This tells the LSTM to ignore PAD tokens
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # Step 3: Run through LSTM
        # output: all hidden states, (h_n, c_n): final hidden/cell states
        packed_output, (h_n, c_n) = self.lstm(packed)
        
        # Step 4: Use the last hidden state of the top LSTM layer
        # h_n shape: (num_layers, batch_size, hidden_dim)
        # We take the last layer: h_n[-1] → (batch_size, hidden_dim)
        hidden = h_n[-1]
        
        # Step 5: Classification
        hidden = self.dropout(hidden)
        output = self.fc(hidden)       # (batch_size, 1)
        output = self.sigmoid(output)  # (batch_size, 1) — probability
        
        return output.squeeze(1)  # (batch_size,)

# Hyperparameters
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2

# Create model
model = LSTMClassifier(
    vocab_size=len(word_to_idx),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    pad_idx=PAD_IDX
).to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

LSTMClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

Total parameters: 3,481,857


## Step 10 — Define Loss Function & Optimizer

- **BCELoss** (Binary Cross-Entropy): Standard loss for binary classification when the model outputs a probability via Sigmoid.
- **Adam optimizer**: Adaptive learning rate, good default for NLP tasks.

In [18]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Loss: BCELoss (Binary Cross-Entropy)")
print(f"Optimizer: Adam (lr=0.001)")

Loss: BCELoss (Binary Cross-Entropy)
Optimizer: Adam (lr=0.001)


## Step 11 — Training Loop

For each epoch:
1. Iterate over batches from our custom DataLoader
2. Unpack the padded sequences, labels, and original lengths
3. Forward pass: model predicts sentiment probability
4. Compute loss (BCE between prediction and true label)
5. Backward pass + weight update
6. Track accuracy: round the probability to 0 or 1 and compare to the true label

In [19]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (sequences, labels, lengths) in enumerate(train_loader):
        # Move data to device
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        # Forward pass
        predictions = model(sequences, lengths)  # (batch_size,)
        loss = criterion(predictions, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping — prevents exploding gradients in RNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Track statistics
        running_loss += loss.item()
        predicted_labels = (predictions >= 0.5).float()
        correct += (predicted_labels == labels).sum().item()
        total += labels.size(0)
        
        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch [{epoch+1}/{num_epochs}], "
                  f"Batch [{batch_idx+1}/{len(train_loader)}], "
                  f"Loss: {loss.item():.4f}")
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] — Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

print("\nTraining complete!")

  Epoch [1/3], Batch [100/390], Loss: 0.6583
  Epoch [1/3], Batch [200/390], Loss: 0.6138
  Epoch [1/3], Batch [300/390], Loss: 0.5879
Epoch [1/3] — Loss: 0.6340, Accuracy: 63.63%
  Epoch [2/3], Batch [100/390], Loss: 0.4178
  Epoch [2/3], Batch [200/390], Loss: 0.5275
  Epoch [2/3], Batch [300/390], Loss: 0.3534
Epoch [2/3] — Loss: 0.4792, Accuracy: 78.62%
  Epoch [3/3], Batch [100/390], Loss: 0.3674
  Epoch [3/3], Batch [200/390], Loss: 0.5682
  Epoch [3/3], Batch [300/390], Loss: 0.2911
Epoch [3/3] — Loss: 0.3658, Accuracy: 84.94%

Training complete!


## Step 12 — Evaluate on Test Set

We evaluate the trained model on the test set to see how well it generalizes to unseen reviews.

In [20]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for sequences, labels, lengths in test_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        predictions = model(sequences, lengths)
        predicted_labels = (predictions >= 0.5).float()
        
        correct += (predicted_labels == labels).sum().item()
        total += labels.size(0)

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 82.73%


## Step 13 — Test on Custom Sentences

Let's see how the model classifies some custom review texts.

In [21]:
def predict_sentiment(text, model, word_to_idx, max_len=256):
    """Predict sentiment for a single text string."""
    model.eval()
    
    # Tokenize and numericalize
    tokens = simple_tokenizer(text)[:max_len]
    indices = [word_to_idx.get(t, UNK_IDX) for t in tokens]
    
    # Create tensors (batch of 1)
    seq_tensor = torch.tensor([indices], dtype=torch.long).to(device)
    length_tensor = torch.tensor([len(indices)])
    
    with torch.no_grad():
        prob = model(seq_tensor, length_tensor).item()
    
    sentiment = "Positive" if prob >= 0.5 else "Negative"
    return sentiment, prob

# Test with custom reviews
test_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the story kept me engaged throughout.",
    "Terrible film. Boring plot, bad acting, and a complete waste of time.",
    "It was okay, nothing special but not terrible either.",
    "One of the best movies I have ever seen. A masterpiece!",
    "I fell asleep halfway through. The worst movie of the year."
]

print("=== Custom Review Predictions ===")
for review in test_reviews:
    sentiment, prob = predict_sentiment(review, model, word_to_idx)
    print(f"\nReview: \"{review[:80]}...\"")
    print(f"  → {sentiment} (confidence: {prob:.4f})")

=== Custom Review Predictions ===

Review: "This movie was absolutely fantastic! The acting was superb and the story kept me..."
  → Positive (confidence: 0.7483)

Review: "Terrible film. Boring plot, bad acting, and a complete waste of time...."
  → Negative (confidence: 0.0126)

Review: "It was okay, nothing special but not terrible either...."
  → Negative (confidence: 0.0280)

Review: "One of the best movies I have ever seen. A masterpiece!..."
  → Positive (confidence: 0.9679)

Review: "I fell asleep halfway through. The worst movie of the year...."
  → Negative (confidence: 0.1317)


## Summary

In this notebook we built a complete **custom data pipeline** and **LSTM classifier**:

1. **Loaded** the IMDB dataset using HuggingFace `datasets`
2. **Built a vocabulary** from scratch (word → index mapping)
3. **Custom `IMDBDataset` class** — handles tokenization, numericalization, and truncation in `__getitem__`
4. **Custom `collate_fn`** — dynamically pads sequences per-batch for efficiency
5. **DataLoader** — combines dataset + collate_fn with batching and shuffling
6. **LSTM model** — Embedding → LSTM (2 layers) → FC → Sigmoid, with packed sequences for efficiency
7. **Trained** for 3 epochs with gradient clipping
8. **Evaluated** on test set and custom reviews

**Key takeaways:**
- Custom Datasets give you full control over preprocessing
- Custom collate functions enable variable-length batching
- `pack_padded_sequence` makes LSTM training efficient by skipping PAD tokens
- Gradient clipping prevents exploding gradients in recurrent models
- LSTMs can capture long-range dependencies due to their gating mechanism